# Fine-tune Llama 3 on a Neurvance dataset — under 10 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skech12/Downloader/blob/main/examples/finetune_llama3_unsloth.ipynb)

This notebook shows the full pipeline:
1. Download a CC0 dataset from Neurvance
2. Load Llama 3 8B with Unsloth (4x faster, 70% less VRAM)
3. Fine-tune with QLoRA
4. Run inference on the trained model

**Runtime:** T4 GPU (free on Google Colab)  
**Time:** ~8–10 minutes end to end  
**Dataset used:** `customer-support-qa` from [neurvance.com](https://neurvance.com)

## Step 1 — Install dependencies

In [ ]:
# Install Unsloth + Neurvance downloader
# Unsloth auto-detects CUDA version and installs the right wheels
%%capture
!pip install unsloth neurvance-downloader
!pip install --upgrade transformers datasets trl peft accelerate bitsandbytes

## Step 2 — Download the dataset from Neurvance

In [ ]:
from neurvance import Neurvance

# Free API key at neurvance.com — or download manually and load from disk
nv = Neurvance(api_key="YOUR_API_KEY")
nv.download("customer-support-qa", output_path="dataset.jsonl")

print("Dataset downloaded: dataset.jsonl")

In [ ]:
from datasets import load_dataset

# Load and preview
dataset = load_dataset("json", data_files="dataset.jsonl", split="train")
print(f"Loaded {len(dataset):,} rows")
print("\nSample row:")
print(dataset[0])

## Step 3 — Load Llama 3 8B with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Can go up to 4096+ with RoPE scaling
dtype = None           # Auto-detect: Float16 for T4/V100, BFloat16 for Ampere+
load_in_4bit = True    # QLoRA — cuts VRAM by ~4x

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3-8B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print(f"Model loaded. Parameters: {model.num_parameters():,}")

In [ ]:
# Add LoRA adapters — only trains ~1-2% of parameters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                          # LoRA rank — 16 is a good default
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,                # 0 is optimised for Unsloth
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% less VRAM
    random_state=42,
)

print("LoRA adapters added.")
model.print_trainable_parameters()

## Step 4 — Format the dataset

In [ ]:
# Format to Llama 3 chat template
# Neurvance datasets come as {instruction, input, output} or {question, answer} — adjust as needed

llama3_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a helpful customer support assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>
{}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
{}<|eot_id|>"""

EOS_TOKEN = tokenizer.eos_token

def format_examples(examples):
    # Adapt these field names to match your dataset schema
    questions = examples.get("question", examples.get("instruction", [""]))
    answers   = examples.get("answer",   examples.get("output",      [""]))
    texts = []
    for q, a in zip(questions, answers):
        text = llama3_prompt.format(q, a) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(format_examples, batched=True)
print(f"Formatted. Sample:\n{dataset[0]['text'][:300]}...")

## Step 5 — Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,              # ~5 min on T4 — bump to 200+ for a real run
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
    ),
)

trainer_stats = trainer.train()
print(f"\nDone! Time: {trainer_stats.metrics['train_runtime']:.0f}s")

## Step 6 — Run inference

In [ ]:
# Switch to inference mode (2x faster)
FastLanguageModel.for_inference(model)

inputs = tokenizer(
    [llama3_prompt.format(
        "My order hasn't arrived after 2 weeks. What should I do?",
        ""  # Leave blank — model fills this in
    )],
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    use_cache=True,
    temperature=0.7,
)

response = tokenizer.batch_decode(outputs[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0]
print("Model response:")
print(response)

## Step 7 — Save the model

In [ ]:
# Save LoRA adapters only (small, fast)
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("Saved LoRA adapters to ./lora_model")

# Optional: merge into full model + push to HuggingFace Hub
# model.save_pretrained_merged("full_model", tokenizer, save_method="merged_16bit")
# model.push_to_hub("your-hf-username/my-finetuned-llama3", token="HF_TOKEN")

---

## What's next?

- Browse the full dataset catalog: [neurvance.com/datasets](https://neurvance.com/datasets)
- Request a dataset: [GitHub Issues](https://github.com/skech12/Downloader/issues/new)
- Swap `customer-support-qa` for any other dataset — same 3-line download, same pipeline
- Want Mistral instead? Replace `unsloth/Meta-Llama-3-8B-Instruct` with `unsloth/mistral-7b-instruct-v0.3`

Built with [Unsloth](https://github.com/unslothai/unsloth) + [Neurvance](https://neurvance.com)